In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

class SUIMDataset(Dataset):
    def __init__(self, root_dir, transform=None, mask_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.mask_transform = mask_transform
        self.images_dir = os.path.join(root_dir, 'images')
        self.masks_dir = os.path.join(root_dir, 'masks')
        self.image_filenames = sorted(os.listdir(self.images_dir))

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, img_name.replace('.jpg', '.png')) # Assuming masks are .png

        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L') # Masks are typically single channel (grayscale)

        # Convert mask to a torch tensor first to apply remap_mask
        mask = torch.from_numpy(np.array(mask)).long()
        mask = remap_mask(mask) # Apply the remapping function

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            # Add channel dim for transform, then remove and ensure long type
            mask = self.mask_transform(mask.unsqueeze(0))

        return image, mask

# Define transformations
image_transform = transforms.Compose([
    transforms.Resize((256, 256)), # Resize images to a common size
    transforms.ToTensor(),         # Convert image to PyTorch tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Lambda(lambda x: x.byte()),
    transforms.ToPILImage(),
    transforms.Resize((256, 256), interpolation=Image.NEAREST), # Resize masks using NEAREST interpolation
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.squeeze(0).long())
])

In [ ]:
# Dataset path from previous execution
dataset_path = '/kaggle/input/q3-stage3-2026/dataset/'

# Create dataset instances
suim_dataset = SUIMDataset(root_dir=dataset_path, transform=image_transform, mask_transform=mask_transform)

# Create DataLoaders
batch_size = 4
data_loader = DataLoader(suim_dataset, batch_size=batch_size, shuffle=True)

print(f"Dataset size: {len(suim_dataset)} samples")
print(f"Number of batches per epoch: {len(data_loader)}")

# Display some images and masks
num_samples_to_display = 3

plt.figure(figsize=(10, num_samples_to_display * 4))
for i, (images, masks) in enumerate(data_loader):
    if i >= num_samples_to_display:
        break

    # Convert tensor to numpy for plotting
    img = images[0].permute(1, 2, 0).numpy() # (C, H, W) -> (H, W, C)
    mask = masks[0].squeeze().numpy()      # (1, H, W) -> (H, W)

    # Denormalize image for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1) # Clip values to [0, 1]

    plt.subplot(num_samples_to_display, 2, 2*i + 1)
    plt.imshow(img)
    plt.title(f'Image {i+1}')
    plt.axis('off')

    plt.subplot(num_samples_to_display, 2, 2*i + 2)
    plt.imshow(mask, cmap='viridis', vmin=0, vmax=7) # Assuming 8 classes (0-7)
    plt.title(f'Mask {i+1}')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!pip install segmentation_models_pytorch
import segmentation_models_pytorch as smp


model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
)

print("U-Net model with efficientnet-b1 encoder created.")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(dataloader, desc="Training"):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(dataloader)

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validation"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item()
    return running_loss / len(dataloader)

print("Training and validation loop functions defined.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

train_losses = []
val_losses = []

print("Starting training...")
for epoch in range(num_epochs):
    train_loss = train_epoch(model, data_loader, criterion, optimizer, device)
    val_loss = validate_epoch(model, data_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("Training complete!")

# Plotting the loss curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np


model.eval()
model.to(device)


suim_dataset_eval = SUIMDataset(root_dir=dataset_path, transform=image_transform, mask_transform=mask_transform)
data_loader_eval = DataLoader(suim_dataset_eval, batch_size=1, shuffle=True)

num_samples_to_visualize = 5

plt.figure(figsize=(15, num_samples_to_visualize * 5))

with torch.no_grad():
    for i, (images, masks) in enumerate(data_loader_eval):
        if i >= num_samples_to_visualize:
            break

        image = images.to(device)
        true_mask = masks.to(device)

        # Make prediction
        output = model(image)
        predicted_mask = torch.argmax(output, dim=1)

        img_np = images[0].permute(1, 2, 0).cpu().numpy()
        true_mask_np = true_mask[0].cpu().squeeze().numpy()
        predicted_mask_np = predicted_mask[0].cpu().numpy()

        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_display = std * img_np + mean
        img_display = np.clip(img_display, 0, 1)

        # Plotting
        plt.subplot(num_samples_to_visualize, 3, 3 * i + 1)
        plt.imshow(img_display)
        plt.title(f'Original Image {i+1}')
        plt.axis('off')

        plt.subplot(num_samples_to_visualize, 3, 3 * i + 2)
        plt.imshow(true_mask_np, cmap='viridis', vmin=0, vmax=7)
        plt.title(f'Ground Truth {i+1}')
        plt.axis('off')

        plt.subplot(num_samples_to_visualize, 3, 3 * i + 3)
        plt.imshow(predicted_mask_np, cmap='viridis', vmin=0, vmax=7)
        plt.title(f'Predicted Mask {i+1}')
        plt.axis('off')

plt.tight_layout()
plt.show()